# 4.5 Difference-in-differences with spillovers

This notebook implements the spillover specification:

$$ Y_{it} = \beta \cdot \text{Post}_t \cdot \text{Treated}_i + \gamma \cdot \text{Post}_t \cdot S_i + \theta_t + \eta_i + \epsilon_{it} $$

where $S_i = \sum_{j \neq i} w_{ij} \text{Treated}_j$ is lab group $i$'s exposure to
treated lab groups, with $w_{ij}$ row-normalized so $S_i \in [0,1]$.

This notebook:
- Creates the network proximity measures $w_{ij}$ and exposure measures $S_i$
- Runs the regressions including the spillovers and creates regression tables

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from itertools import combinations
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config
import pyfixest as pf

sys.path.append(str(Path.cwd().parents[0] / "functions"))
from make_regression_table import make_regression_table
from spillover_helpers import normalize_weights, compute_exposure, build_membership_weights

In [2]:
# Load data

# Cleaned lab group data
df = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""]  # Only treat empty strings as NaN
)

# Publications data
publications = pd.read_csv(
    config.PUBLICATON_DATA / 
    "2_Processed" / 
    "publications_matched.csv"
)

# Distances data
distances = pd.read_csv(
    config.CLEAN_DATA / "distances_cleaned.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""]  # Only treat empty strings as missing
)

# Sharing equipment groups data
share_equip_groups = pd.read_csv(
    config.CLEAN_DATA / "share_equip_groups_cleaned.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""]  # Only treat empty strings as missing
)

# Sharing space groups data
share_space_groups = pd.read_csv(
    config.CLEAN_DATA / "share_space_groups_cleaned.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""]  # Only treat empty strings as missing
)

# Communication groups data
comm_groups = pd.read_csv(
    config.CLEAN_DATA / "comm_groups_cleaned.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""]  # Only treat empty strings as missing
)

In [3]:
# Construct post variable
df["post"] = (df["survey"] == "EL").astype(int)

# Keep only labgroups that have both pre and post observations
labgroup_counts = df.groupby("labgroupid")["survey"].nunique()
labgroups_to_keep = labgroup_counts[labgroup_counts == 2].index
df = df[df["labgroupid"].isin(labgroups_to_keep)].copy()

## (1) Constructing the network proximity measures

### (a) Research collaboration

We build $S_i$ for the research-collaboration measure using co-authorship between lab groups. We do this in the following way:

(i) We restrict publications to those dated before the first lab group's BL survey.

(ii) We then construct weights. Weights are defined as: $w_{ij}$ = (publications shared with $j$) / (total publications shared with all $j' \neq i$).

(iii) We then construct $S_i$: $S_i = \sum_{j \neq i} w_{ij} \text{Treated}_j$. If all lab group $i$'s co-authored publications are with treated lab groups, $S_i = 1$; if all lab group $i$'s co-authored publications are with control lab groups, $S_i = 0$. Lab groups with no co-authorship publications get $S_i = 0$.

Note: date in the publications data is sometimes year-only. As a robustness check, we exclude these publications.

In [4]:
# (i) Restrict publications to those dated before the first lab group's BL survey

# Pre-treatment cutoff: first lab group's BL survey date
cutoff_date = pd.to_datetime(df.loc[df["survey"] == "BL", "survey_date_bl"]).min()
print(f"Pre-treatment cutoff for network construction: {cutoff_date.date()}")


# Create clean date column. "mixed" required as date mixes year only and full dates
publications["date_parsed"] = pd.to_datetime(publications["date"], errors="coerce", format="mixed")

# Restrict to publications that are pre-treatment and link 2+ lab groups
pre_treatment_pubs = publications[
    (publications["date_parsed"] < cutoff_date) & (publications["n_matched_labgroupids"] > 1)
].copy()

print(f"{len(pre_treatment_pubs)} of {len(publications)} publications are pre-treatment and link 2+ lab groups")

Pre-treatment cutoff for network construction: 2025-10-06
396 of 8014 publications are pre-treatment and link 2+ lab groups


In [5]:
# (ii) Create weights

# Build edge weights: for each pre-treatment publication matched to 2+ labs,
# add 1 to every pair of matched labgroupids
edge_counts = {}
for ids in pre_treatment_pubs["matched_labgroupids"]:
    lab_ids = sorted(set(int(x) for x in str(ids).split(";")))
    for a, b in combinations(lab_ids, 2):
        edge_counts[(a, b)] = edge_counts.get((a, b), 0) + 1

edges = pd.DataFrame(
    [(a, b, w) for (a, b), w in edge_counts.items()],
    columns=["labgroupid_a", "labgroupid_b", "n_shared_pubs"]
)
print(f"{len(edges)} lab-group pairs share at least one pre-treatment publication")

# Restrict the network's node set to lab groups that consented to data merging
consented = df["consent_data_merge"] == "Yes I consent to this data collection and merging"
consenting_labgroupids = df.loc[consented, "labgroupid"].unique()
n_excluded = df["labgroupid"].nunique() - len(consenting_labgroupids)
print(f"{n_excluded} of {df['labgroupid'].nunique()} lab groups did not consent to publication data merging and are excluded from the collaboration network")

W_collaboration = normalize_weights(
    edges, id_col_i="labgroupid_a", id_col_j="labgroupid_b",
    weight_col="n_shared_pubs", all_ids=consenting_labgroupids
)

67 lab-group pairs share at least one pre-treatment publication
28 of 109 lab groups did not consent to publication data merging and are excluded from the collaboration network


In [6]:
# (iii) Compute S_i

# Compute S_i = sum_j W_ij * D_j, where D_j is the treatment status of lab j
treated_by_lab = df.drop_duplicates("labgroupid").set_index("labgroupid")["treated"]
S_collaboration = compute_exposure(W_collaboration, treated_by_lab)

# Non-consenting labs get S_collaboration = NaN (dropped in regressions below)
df["S_collaboration"] = df["labgroupid"].map(S_collaboration)
assert df["S_collaboration"].dropna().between(0, 1).all()

# Report distribution of S_collaboration
print(df["S_collaboration"].describe())
n_zero = (S_collaboration == 0).sum()
print(f"{n_zero} of {len(S_collaboration)} consenting lab groups have zero measured collaboration exposure")

count    162.000000
mean       0.234053
std        0.381328
min        0.000000
25%        0.000000
50%        0.000000
75%        0.500000
max        1.000000
Name: S_collaboration, dtype: float64
55 of 81 consenting lab groups have zero measured collaboration exposure


### (b) Geographical distance measure construction
We build $S_i$ for the geographical distance measure using the pairwise building distance computed
in 1_Cleaning/6_3_create_distance_measure.ipynb (0 if the pair shares a building, otherwise the
haversine distance in km between their buildings). We do this in the following way:

(i)-(ii) We construct weights as $w_{ij} = \frac{1}{(1 + d_{ij})^2}$, where $d_{ij}$ is the distance
in km between $i$ and $j$'s buildings - same-building pairs get the maximum weight of 1, decaying
quickly with distance (squared rather than linear, since $1/(1+d)$ left $S_i$ with almost no
cross-lab variation on such a compact campus). Unlike the binary group-sharing measures, every
lab-group pair gets a strictly positive weight (before row-normalization), since this is a dense
network rather than one defined by discrete ties.

(iii) We then construct $S_i$: $S_i = \sum_{j \neq i} w_{ij} \text{Treated}_j$. Lab groups with no
room data have no computable distance to anyone, so $S_i$ is missing (not 0) for them, and they are
dropped from the distance spillover regressions - same treatment as non-consenting labs for
$S_{collaboration}$.

In [7]:
# (i)-(ii) Build weights: w_ij = 1 / (1 + distance_km)^2. distances_cleaned.csv has each unordered
# pair once; normalize_weights' symmetric=True (default) mirrors it to both w_ij and w_ji.
distances["weight"] = 1 / (1 + distances["distance_km"]) ** 2

# Restrict the network's node set to lab groups with room data
labs_with_room_data = set(distances["labgroupid_a"]) | set(distances["labgroupid_b"])
panel_labs_with_room_data = [lg for lg in df["labgroupid"].unique() if lg in labs_with_room_data]
n_excluded = df["labgroupid"].nunique() - len(panel_labs_with_room_data)
#print(f"{n_excluded} of {df['labgroupid'].nunique()} lab groups have no room data and are excluded from the distance network")

W_distance = normalize_weights(
    distances, id_col_i="labgroupid_a", id_col_j="labgroupid_b",
    weight_col="weight", all_ids=panel_labs_with_room_data
)

In [8]:
# (iii) Compute S_i
S_distance = compute_exposure(W_distance, treated_by_lab)
df["S_distance"] = df["labgroupid"].map(S_distance)  # NaN for labs with no room data (not in W_distance)

n_missing = df["S_distance"].isna().sum()
# print(f"{n_missing} of {len(df)} panel rows have no room data and get S_distance = NaN")

assert df["S_distance"].dropna().between(0, 1).all()
print(df["S_distance"].describe())
n_zero = (df["S_distance"] == 0).sum()
print(f"{n_zero} of {df['S_distance'].notna().sum()} lab groups with room data have zero measured distance exposure")

count    212.000000
mean       0.524537
std        0.055264
min        0.312436
25%        0.525268
50%        0.533358
75%        0.541895
max        0.697944
Name: S_distance, dtype: float64
0 of 212 lab groups with room data have zero measured distance exposure


### (c) Equipment sharing measure construction
We build $S_i$ for the equipment sharing measure in the following way:

(i) We restrict equipment sharing groups to those in our sample. We remove any entries that are self=referencing.

(ii) We then construct weights. Weights are defined as: $w_{ij} = 1$ if $i$ and $j$ form a labgroupid, sample_group pair, 0 if not.

(iii) We then construct $S_i$: $S_i = \sum_{j \neq i} w_{ij} \text{Treated}_j$. If all lab group $i$'s equipment sharing groups are treated lab groups, $S_i = 1$; if all lab group $i$'s equipment sharing groups are control lab groups, $S_i = 0$. Lab groups with no equipment sharing groups in the sample get $S_i = 0$.

In [9]:
# (i)-(ii) Restrict to non-missing sample_group and build weights
n_self = (share_equip_groups["labgroupid"] == share_equip_groups["sample_group"]).sum()
n_dup = share_equip_groups.dropna(subset=["sample_group"]).duplicated(subset=["labgroupid", "sample_group"]).sum()
print(f"{n_self} self-referencing rows and {n_dup} duplicate (labgroupid, sample_group) pairs dropped")

W_equip = build_membership_weights(share_equip_groups, all_ids=df["labgroupid"].unique())

26 self-referencing rows and 26 duplicate (labgroupid, sample_group) pairs dropped


In [10]:
# (iii) Compute S_i
S_equip = compute_exposure(W_equip, treated_by_lab)
df["S_equip"] = df["labgroupid"].map(S_equip)
assert df["S_equip"].between(0, 1).all()

print(df["S_equip"].describe())
print(f"{(S_equip == 0).sum()} of {len(S_equip)} lab groups have zero measured equipment-sharing exposure")

count    218.000000
mean       0.363692
std        0.398311
min        0.000000
25%        0.000000
50%        0.250000
75%        0.666667
max        1.000000
Name: S_equip, dtype: float64
53 of 109 lab groups have zero measured equipment-sharing exposure


### (d) Space sharing measure construction
We build $S_i$ for the space sharing measure the same way as equipment sharing:

(i) We restrict space sharing groups to those in our sample.

(ii) We then construct weights. Weights are defined as: $w_{ij} = 1$ if $i$ and $j$ form a labgroupid, sample_group pair, 0 if not.

(iii) We then construct $S_i$: $S_i = \sum_{j \neq i} w_{ij} \text{Treated}_j$. If all lab group $i$'s space sharing groups are treated lab groups, $S_i = 1$; if all lab group $i$'s space sharing groups are control lab groups, $S_i = 0$. Lab groups with no space sharing groups in the sample get $S_i = 0$.

In [11]:
# (i)-(ii) Restrict to non-missing sample_group and build weights
n_self = (share_space_groups["labgroupid"] == share_space_groups["sample_group"]).sum()
n_dup = share_space_groups.dropna(subset=["sample_group"]).duplicated(subset=["labgroupid", "sample_group"]).sum()
print(f"{n_self} self-referencing rows and {n_dup} duplicate (labgroupid, sample_group) pairs dropped")

W_space = build_membership_weights(share_space_groups, all_ids=df["labgroupid"].unique())

18 self-referencing rows and 6 duplicate (labgroupid, sample_group) pairs dropped


In [12]:
# (iii) Compute S_i
S_space = compute_exposure(W_space, treated_by_lab)
df["S_space"] = df["labgroupid"].map(S_space)
assert df["S_space"].between(0, 1).all()

print(df["S_space"].describe())
print(f"{(S_space == 0).sum()} of {len(S_space)} lab groups have zero measured space-sharing exposure")

count    218.000000
mean       0.294656
std        0.400519
min        0.000000
25%        0.000000
50%        0.000000
75%        0.571429
max        1.000000
Name: S_space, dtype: float64
66 of 109 lab groups have zero measured space-sharing exposure


### (e) Communication measure construction
We build $S_i$ for the communication measure the same way as equipment and space sharing:

(i) We restrict communication groups to those in our sample.

(ii) We then construct weights. Weights are defined as: $w_{ij} = 1$ if $i$ and $j$ form a labgroupid, sample_group pair, 0 if not.

(iii) We then construct $S_i$: $S_i = \sum_{j \neq i} w_{ij} \text{Treated}_j$. If all lab group $i$'s communication groups are treated lab groups, $S_i = 1$; if all lab group $i$'s communication groups are control lab groups, $S_i = 0$. Lab groups with no communication groups in the sample get $S_i = 0$.

In [13]:
# (i)-(ii) Restrict to non-missing sample_group and build weights
n_self = (comm_groups["labgroupid"] == comm_groups["sample_group"]).sum()
n_dup = comm_groups.dropna(subset=["sample_group"]).duplicated(subset=["labgroupid", "sample_group"]).sum()
print(f"{n_self} self-referencing rows and {n_dup} duplicate (labgroupid, sample_group) pairs dropped")

W_comm = build_membership_weights(comm_groups, all_ids=df["labgroupid"].unique())

13 self-referencing rows and 23 duplicate (labgroupid, sample_group) pairs dropped


In [14]:
# (iii) Compute S_i
S_comm = compute_exposure(W_comm, treated_by_lab)
df["S_comm"] = df["labgroupid"].map(S_comm)
assert df["S_comm"].between(0, 1).all()

print(df["S_comm"].describe())
print(f"{(S_comm == 0).sum()} of {len(S_comm)} lab groups have zero measured communication exposure")

count    218.000000
mean       0.347925
std        0.429510
min        0.000000
25%        0.000000
50%        0.000000
75%        0.800000
max        1.000000
Name: S_comm, dtype: float64
62 of 109 lab groups have zero measured communication exposure


## (2) Difference-in-differences with spillovers

We estimate the PAP spec for the primary outcome (levels and log), plus a version adding the triple interaction $\text{Post}_t \cdot \text{Treated}_i \cdot S_i$, which lets the spillover effect differ for treated vs. control lab groups.

### (a) Research collaboration

Note: since some lab groups did not consent to publication data merging, these regressions run on a smaller sample.

In [15]:
n_dropped = df["S_collaboration"].isna().sum()
print(f"{n_dropped} of {len(df)} panel rows have missing S_collaboration and will be dropped from the spillover regressions")

df["log_electricity"] = np.log1p(df["annual_electricity_total"])

# Same-sample baseline: plain treated:post, no spillover terms, restricted to 
# the same consenting-lab sample as the spillover specs below
df_consenting = df.dropna(subset=["S_collaboration"])

fit_levels_baseline = pf.feols(
    "annual_electricity_total ~ treated:post | labgroupid + post",
    data=df_consenting, vcov={"CRV1": "labgroupid"}
)
fit_log_baseline = pf.feols(
    "log_electricity ~ treated:post | labgroupid + post",
    data=df_consenting, vcov={"CRV1": "labgroupid"}
)

fit_levels_main = pf.feols(
    "annual_electricity_total ~ treated:post + S_collaboration:post | labgroupid + post",
    data=df, vcov={"CRV1": "labgroupid"}
)
fit_levels_triple = pf.feols(
    "annual_electricity_total ~ treated:post + S_collaboration:post + treated:S_collaboration:post | labgroupid + post",
    data=df, vcov={"CRV1": "labgroupid"}
)
fit_log_main = pf.feols(
    "log_electricity ~ treated:post + S_collaboration:post | labgroupid + post",
    data=df, vcov={"CRV1": "labgroupid"}
)
fit_log_triple = pf.feols(
    "log_electricity ~ treated:post + S_collaboration:post + treated:S_collaboration:post | labgroupid + post",
    data=df, vcov={"CRV1": "labgroupid"}
)

fit_levels_main.summary()

56 of 218 panel rows have missing S_collaboration and will be dropped from the spillover regressions


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


###

Estimation:  OLS
Dep. var.: annual_electricity_total, Fixed effects: labgroupid + post
sample: None = all
Inference:  CRV1
Observations:  162

| Coefficient          |   Estimate |   Std. Error |   t value |   Pr(>|t|) |     2.5% |   97.5% |
|:---------------------|-----------:|-------------:|----------:|-----------:|---------:|--------:|
| treated:post         |     26.420 |       96.437 |     0.274 |      0.785 | -165.495 | 218.335 |
| S_collaboration:post |   -136.077 |       77.591 |    -1.754 |      0.083 | -290.488 |  18.334 |
---
RMSE: 219.708 R2: 1.0 R2 Within: 0.015 


In [16]:
# Export to nice table
table = make_regression_table(
    fit_list = [
        fit_levels_baseline, fit_levels_main, fit_levels_triple,
        fit_log_baseline, fit_log_main, fit_log_triple,
    ],
    model_names   = ["(1)", "(2)", "(3)", "(4)", "(5)", "(6)"],
    keep_vars     = ["treated:post", "S_collaboration:post", "treated:S_collaboration:post"],
    var_labels    = {
        "treated:post": "Treated $\\times$ Post",
        "S_collaboration:post": "Exposure (collab.) $\\times$ Post",
        "treated:S_collaboration:post": "Treated $\\times$ Exposure $\\times$ Post",
    },
    fe_rows       = {
        "Research group FE": [True] * 6,
        "Time FE":            [True] * 6,
    },
    col_groups    = {"Levels": [0, 1, 2], "Log": [3, 4, 5]},
    col_subgroups = {"Baseline": [0, 3], "Main": [1, 4], "Triple interaction": [2, 5]},
    baseline_mean  = "auto",
    outcome_levels = "annual_electricity_total",
    df_levels      = df,
    decimals       = [1, 1, 1, 3, 3, 3],
    mean_decimals  = [0, 0, 0, 3, 3, 3],
    r2_type        = None,
    col1_width     = "5.5cm",
    coln_width     = "2.1cm",
)
table_path = config.OUTPUT / "5_Regression_Tables" / "spillovers_collaboration.tex"
_ = table_path.write_text(table)

### (b) Comparing exposure measures

We compare the Main spec across all available proximity measures, one table for levels and one for
log (no triple interaction here - see the collaboration-only table above for that). The first column
is a plain treated:post spec with no exposure term, as a reference point; the remaining columns follow
the PAP's own ordering of proximity measures (distance, space, equipment, communication, collaboration).

Note: $S_{collaboration}$ is only defined for labs that consented to publication data merging, and
$S_{distance}$ only for labs with room data - both columns therefore run on a smaller sample than the
other measures - see "Number of observations".

In [17]:
# Fit the Main spec for one measure, on one outcome.
# The exposure column is renamed to "S" so every measure's fit shares the same coefficient
# name (S:post) - this is what lets one table row span all measures below.
def fit_measure(outcome, exposure_col, data=df):
    d = data.dropna(subset=[exposure_col]).rename(columns={exposure_col: "S"})
    return pf.feols(f"{outcome} ~ treated:post + S:post | labgroupid + post", data=d, vcov={"CRV1": "labgroupid"})

# Plain treated:post, no exposure term - reference column before controlling for any measure
def fit_baseline(outcome, data=df):
    return pf.feols(f"{outcome} ~ treated:post | labgroupid + post", data=data, vcov={"CRV1": "labgroupid"})

# Column order follows the PAP's own listing of proximity measures.
measures = {
    "Distance": ("S_distance", W_distance),
    "Space": ("S_space", W_space),
    "Equipment": ("S_equip", W_equip),
    "Communication": ("S_comm", W_comm),
    "Collaboration": ("S_collaboration", W_collaboration),
}

fits_levels = [fit_baseline("annual_electricity_total")] + [
    fit_measure("annual_electricity_total", col) for col, _ in measures.values()
]
fits_log = [fit_baseline("log_electricity")] + [
    fit_measure("log_electricity", col) for col, _ in measures.values()
]
model_names = [f"({i + 1})" for i in range(len(measures) + 1)]

# Column 0 is Baseline, left out of "Proximity measures" and given its own header cell
col_groups = {"Proximity measures": list(range(1, len(measures) + 1))}
col_subgroups = {"Baseline": [0], **{name: [i + 1] for i, name in enumerate(measures)}}

In [18]:
# Export the combined tables (levels and log)
common_kwargs = dict(
    model_names=model_names,
    keep_vars=["treated:post", "S:post"],
    var_labels={
        "treated:post": "Treated $\\times$ Post",
        "S:post": "Exposure $\\times$ Post",
    },
    col_groups=col_groups,
    col_subgroups=col_subgroups,
    r2_type=None,
    outcome_levels="annual_electricity_total",
    df_levels=df,
    col1_width="5.5cm",
    coln_width="2.8cm",
)

table_levels = make_regression_table(
    fit_list=fits_levels, baseline_mean="auto", decimals=1, mean_decimals=0, **common_kwargs,
)
(config.OUTPUT / "5_Regression_Tables" / "spillovers_by_measure_levels.tex").write_text(table_levels)

table_log = make_regression_table(
    fit_list=fits_log, baseline_mean="auto", decimals=3, mean_decimals=3, **common_kwargs,
)
(config.OUTPUT / "5_Regression_Tables" / "spillovers_by_measure_log.tex").write_text(table_log)

1096